In [2]:
from langchain_community.document_loaders import WebBaseLoader
import bs4
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Only keep post title, headers, and content from the full HTML.
bs4_strainer = bs4.SoupStrainer(class_=("post-title", "post-header", "post-content"))
# 加载器
loader = WebBaseLoader(
    web_paths=["https://lilianweng.github.io/posts/2023-06-23-agent/"],
    bs_kwargs={"parse_only": bs4_strainer},
)
# 加载文档
docs = loader.load()
print(len(docs))
print(f"Total characters: {len(docs[0].page_content)}")

# 打印前500字符
if len(docs[0].page_content) > 500 :
    print(docs[0].page_content[:500])

1
Total characters: 43047


      LLM Powered Autonomous Agents
    
Date: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng


Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.
Agent System Overview#
In


In [3]:
# 创建文本拆分器，1000字符/块，重叠200字符并设置开始索引
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    add_start_index=True
)
# 分割文档
all_splits = text_splitter.split_documents(docs)
print(len(all_splits))


63


In [11]:
# 将分块数据使用嵌入模型转换为向量数据存醋
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(model="nomic-embed-text")
vectorstore = Chroma(embedding_function=embeddings)
document_ids = vectorstore.add_documents(documents=all_splits)
print(document_ids[:3])

['6a86c605-724d-48c9-9770-0b3885163c11', '25168475-12a1-48e8-8da7-4def2212250c', '1dd9bc79-f0fe-4169-b991-ea6a72fc2d7c']


In [13]:
# 使用向量存储构建检索器，并设置搜索类型和关键字参数
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 6})
# 检索文档关联内容
retrieved_docs = retriever.invoke("What are the approaches to Task Decomposition?")
print(len(retrieved_docs))
print(retrieved_docs[0].page_content)

6
Component One: Planning#
A complicated task usually involves many steps. An agent needs to know what they are and plan ahead.
Task Decomposition#
Chain of thought (CoT; Wei et al. 2022) has become a standard prompting technique for enhancing model performance on complex tasks. The model is instructed to “think step by step” to utilize more test-time computation to decompose hard tasks into smaller and simpler steps. CoT transforms big tasks into multiple manageable tasks and shed lights into an interpretation of the model’s thinking process.
Tree of Thoughts (Yao et al. 2023) extends CoT by exploring multiple reasoning possibilities at each step. It first decomposes the problem into multiple thought steps and generates multiple thoughts per step, creating a tree structure. The search process can be BFS (breadth-first search) or DFS (depth-first search) with each state evaluated by a classifier (via a prompt) or majority vote.


In [23]:
import requests
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
import os

load_dotenv()
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')
if not deepseek_api_key:
    raise ValueError("DEEPSEEK_API_KEY 环境变量未设置")

base_url = "https://api.deepseek.com/v1"

def list_models(api_key):
    headers = {"Authorization": f"Bearer {api_key}"}
    response = requests.get("https://api.deepseek.com/v1/models", headers=headers)
    return response.json()

# 使用
models = list_models(os.getenv("DEEPSEEK_API_KEY"))
print("可用模型:", [model["id"] for model in models["data"]])

可用模型: ['deepseek-chat', 'deepseek-reasoner']


In [ ]:
model = ChatOpenAI(
    base_url=base_url,
    api_key=deepseek_api_key,
    model="deepseek-chat",
)
# model.invoke("你是谁？").content

In [45]:
from langchain.chat_models import init_chat_model
llm = init_chat_model(model="llama3", model_provider="ollama")
llm.invoke([("human","你是谁？"), {"role": "system", "content": "使用简体中文回答"}]).content

'我是 LLaMA，一个由 Meta 开发的基于人工智能技术的聊天机器人。我的目的是帮助用户交流、提供信息和答案，或者只是进行有趣的对话。我不具有个人的经历、背景或身份，但我会尽力回答你的问题和帮助你解决问题。'

In [50]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model = "llama3",
    temperature = 0.8,
    num_predict = 256,
    # other params ...
)
messages = [ ("system", "使用中文回答"), ("human", "你是谁？")]
for chunk in llm.stream(messages):
    print(chunk.text(), end="")

我是 LLaMA，一个由 Meta AI 开发的基于人工智能的语言模型。我可以理解和生成人类语言，帮助用户回答问题、创作内容或进行对话。我的能力包括：

* 了解自然语言处理（NLP）中的基本概念和技术
* 能够生成文本、对话和回答问题
* 具有良好的语言理解能力，可以识别语义和意图

我可以用来帮助用户完成各种任务，例如：

* 回答问题
* 创作故事或诗歌
* 生成对话或对话脚本
* 帮助用户学习新语言或文化
* 等等

如果你有任何问题或需要我的帮助，请随时问我！

In [27]:
# 使用LangChain hud的 RAG提示符
from langchain import hub

prompt = hub.pull("rlm/rag-prompt")
example_msgs = prompt.invoke({
    "context": "filler context",
    "question": "filler question",
}).to_messages()
print(example_msgs)

[HumanMessage(content="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: filler question \nContext: filler context \nAnswer:", additional_kwargs={}, response_metadata={})]


In [28]:
if len(example_msgs) > 0:
    print(example_msgs[0].content)

You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.
Question: filler question 
Context: filler context 
Answer:


In [47]:
# Runnable协议的链式调用
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# 链式调用，存储器-格式化文档函数
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

for chunk in rag_chain.stream("What is Task Decomposition? 并将其翻译成中文"):
    print(chunk, end="", flush=True)

Task decomposition refers to the process of breaking down complex tasks into smaller, more manageable steps. This can be done through various approaches, such as using language models (LLM) with simple prompting or relying on external classical planners.

中文翻译：任务分解是将复杂的任务分解成更小、更易管理的步骤。可以通过各种方法实现，例如使用语言模型（LLM）进行简单的提示或依靠外部经典规划器。

In [53]:
# LangChain中的内置链
from langchain.chains.retrieval import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)
prompt = ChatPromptTemplate.from_messages([("system", system_prompt),("human", "{input}"),])

question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)
response = rag_chain.invoke({"input": "What is Task Decomposition? 并翻译成中文"})
print(response["answer"])
print("--------------------------------")
# 向用户展示用于生成答案的来源
for document in response["context"]:
    print(document)
    print()

According to the provided context, Task Decomposition refers to the process of breaking down complex tasks into smaller, more manageable steps. This can be achieved through various methods, such as:

1. Chain of Thought (CoT): instructing a model to "think step by step" to decompose hard tasks into smaller and simpler steps.
2. Tree of Thoughts: extending CoT by exploring multiple reasoning possibilities at each step, creating a tree structure.

Task Decomposition can be done through various approaches, including:

1. Large Language Models (LLMs) with simple prompting
2. Task-specific instructions
3. Human inputs

Translation:
任务分解（Task Decomposition）是将复杂的任务分解成更小、更可管理的步骤。这种方法可以通过以下几种方式实现：

1.链式思想（CoT）： instructing a model to "think step by step" to decompose hard tasks into smaller and simpler steps。
2.思想树：扩展 CoT，探索每个步骤中的多种推理可能性，创建一个树结构。

任务分解可以通过以下几种方法实现：

1.大语言模型（LLMs）与简单的提示
2.特定任务
--------------------------------
page_content='Component One: Planning#
A complicated task usually invo